In [107]:
import pandas as pd
import numpy as np

In [108]:
df = pd.read_csv("C:/Users/Mai/VSC Projects/Clinical Trial Opportunity Radar/tests/opportunities_ds.csv")

display(df.dtypes)

country           str
therap_area       str
indication        str
patient_pop     int64
unmet_need      int64
trial_compet    int64
op_feas         int64
reg_env         int64
dtype: object

In [109]:
print("Dataset dimensions:", df.shape)
display(df.head())
display(df.dtypes)

Dataset dimensions: (135, 8)


,country,therap_area,indication,patient_pop,unmet_need,trial_compet,op_feas,reg_env
0,Argentina,Neurology,Multiple sclerosis,11895,71,5,58,70
1,Argentina,Neurology,Parkinson's disease,17978,85,5,61,70
2,Argentina,Neurology,Amyotrophic lateral sclerosis,4487,97,3,42,70
3,Argentina,Rheumatology,Rheumatoid arthritis,27106,62,7,65,70
4,Argentina,Rheumatology,Systemic lupus erythematosus,16166,84,4,42,70


country           str
therap_area       str
indication        str
patient_pop     int64
unmet_need      int64
trial_compet    int64
op_feas         int64
reg_env         int64
dtype: object

In [110]:
#weights definition
weights = {
    "patient_pop": 0.40,
    "unmet_need": 0.15,
    "op_feas": 0.10,
    "reg_env": 0.20,
    "trial_compet": 0.15
}

print("Total weight:", sum(weights.values()))

Total weight: 1.0


In [111]:
positive_vars = [
    "patient_pop",
    "unmet_need",
    "op_feas",
    "reg_env"
]

negative_vars = [
    "trial_compet"
]

In [112]:
score_vars = list(weights.keys())

display(
    df[score_vars]
    .describe()
    .T
    .round(2)
)

,count,mean,std,min,25%,50%,75%,max
patient_pop,135.0,20503.61,19543.71,1000.0,6606.5,14497.0,27826.0,100000.0
unmet_need,135.0,67.75,16.87,28.0,57.0,68.0,81.0,100.0
op_feas,135.0,56.08,9.34,31.0,50.0,57.0,63.0,74.0
reg_env,135.0,66.00,7.76,50.0,62.0,68.0,72.0,75.0
trial_compet,135.0,4.12,1.91,1.0,3.0,4.0,6.0,7.0


Normalization


In [113]:
# positive
for var in positive_vars:
    df[f"{var}_norm"] = (
        (df[var] - df[var].min())
        / (df[var].max() - df[var].min())
    )

In [114]:
#neg var = "trial_compet"

df[f"{var}_norm"] = (
    (df[var].max() - df[var])
    / (df[var].max() - df[var].min())
)

Scoring


In [115]:
var = "trial_compet"

df[f"{var}_norm"] = (
    (df[var].max() - df[var])
    / (df[var].max() - df[var].min())
)

In [116]:
df["opportunity_score"] = sum(
    df[f"{var}_norm"] * weight
    for var, weight in weights.items()
) * 100

In [117]:
df["opportunity_score"] = df["opportunity_score"].round(2)

display(
    df[
        ["country", "therap_area", "indication", "opportunity_score"]
    ]
    .sort_values("opportunity_score", ascending=True)
    .head(15)
)

,country,therap_area,indication,opportunity_score
119,Uruguay,Dermatology,Acne vulgaris,19.74
36,Chile,Oncology,Non-small cell lung cancer,19.90
37,Chile,Oncology,Breast cancer,20.14
30,Chile,Neurology,Multiple sclerosis,21.70
43,Chile,Dermatology,Plaque psoriasis,22.53
39,Chile,Respiratory,Asthma,23.23
114,Uruguay,Respiratory,Asthma,23.64
33,Chile,Rheumatology,Rheumatoid arthritis,23.66
44,Chile,Dermatology,Acne vulgaris,24.37
112,Uruguay,Oncology,Breast cancer,24.67


In [118]:
print("Score range:", df["opportunity_score"].min(), "-", df["opportunity_score"].max())
print("Missing scores:", df["opportunity_score"].isna().sum())

display(df["opportunity_score"].describe().round(2))

Score range: 19.74 - 53.62
Missing scores: 0


count    135.00
mean      36.40
std        7.95
min       19.74
25%       30.78
50%       36.80
75%       41.39
max       53.62
Name: opportunity_score, dtype: float64

Validation 

In [119]:
assert df["opportunity_score"].between(0, 100).all()
assert df["opportunity_score"].notna().all()
assert len(df) == 135

print("All validation checks passed.")

All validation checks passed.


In [120]:
top_by_country = (
    df.sort_values("opportunity_score", ascending=False)
    .groupby("country", sort=False)
    .head(3)
    .sort_values("opportunity_score", ascending=False)
)

display(
    top_by_country[
        ["country", "therap_area", "indication", "opportunity_score"]
    ]
)

,country,therap_area,indication,opportunity_score
134,Venezuela,Dermatology,Acne vulgaris,53.62
24,Brazil,Respiratory,Asthma,52.92
29,Brazil,Dermatology,Acne vulgaris,52.83
130,Venezuela,Respiratory,Chronic obstructive pulmonary disease,52.72
132,Venezuela,Dermatology,Atopic dermatitis,52.18
25,Brazil,Respiratory,Chronic obstructive pulmonary disease,50.44
87,Mexico,Dermatology,Atopic dermatitis,49.60
84,Mexico,Respiratory,Asthma,47.71
94,Paraguay,Rheumatology,Systemic lupus erythematosus,47.26
92,Paraguay,Neurology,Amyotrophic lateral sclerosis,44.30


In [121]:
df.to_csv("../opportunities_scored.csv", index=False)

print("Scored dataset saved as opportunities_scored.csv")

Scored dataset saved as opportunities_scored.csv
